
# **FACULTAD DE INGENIERÍA - BIOINGENIERÍA**
# **Bioseñales y Sistemas**
## **Segundo Proyecto - Clasificación de Datos BCI**

*Maria José Rios Hurtado*

*Mónica Alejandra Hinestroza Chaparro*


### **1. CONSULTA**

Consultar qué índices del EEG, adicionales a los realizados en el proyecto 1 basados en densidad espectral de potencia, mejoran la clasificación de estados en BCI. Citar las fuentes 
Escoger por lo menos tres índices que se puedan implementar en Python, se pueden usar librerías, explicar la medida y su uso en los datos del proyecto.

<div align = "justify">

La densidad espectral de potencia (PSD) es uno de los métodos más empleados para caracterizar señales EEG en sistemas BCI basados en imaginería motora, pero su capacidad discriminativa tiene límites: opera por canal de forma independiente, asume cierta estacionariedad y no captura la distribución espacial ni la complejidad temporal de la actividad cortical. La literatura reciente identifica varios índices complementarios que mejoran la precisión de clasificación cuando se combinan o reemplazan a la PSD.


En el Proyecto 1 se caracterizó estados motores (reposo, movimiento real e imaginación) mediante la densidad espectral de potencia (PSD) en las bandas mu (8–13 Hz) y beta (13–30 Hz) sobre los canales C3 y C4. Si bien la PSD captura información sobre la energía por frecuencia, no refleja la distribución espacial de la actividad cortical, la dinámica temporal de la señal ni su complejidad no lineal. Los tres índices presentados a continuación complementan directamente los hallazgos de ese proyecto, son ampliamente respaldados por la literatura BCI y son implementables en Python.

</div>

<div align ="justify">

#### **Common Spatial Pattern (CSP)**

El CSP es una técnica de filtrado espacial que mejora la capacidad discriminativa de las señales EEG al ser particularmente útil para problemas de clasificación binaria, como distinguir entre imaginación de movimiento de mano derecha e izquierda. El CSP opera encontrando filtros espaciales que maximizan la varianza para una clase mientras la minimizan para la otra. Este principio lo hace directamente complementario a los análisis PSD del Proyecto 1, donde la lateralización entre C3 y C4 fue el hallazgo central: mientras la PSD cuantifica la potencia en cada canal por separado, el CSP encuentra la combinación lineal de todos los canales que maximiza esa diferencia inter-hemisférica de forma óptima.[1]

Durante la imaginería motora, los ritmos sensoriomotores se atenúan y luego se amplifican en corto tiempo, fenómeno conocido como desincronización/sincronización relacionada con eventos (ERD/ERS). Para extraer de forma óptima las características EEG que describen este fenómeno, el algoritmo CSP busca filtros espaciales que extraigan características espaciales discriminativas entre clases y es adoptado frecuentemente debido a su buen desempeño. Sin embargo, el desempeño del CSP depende fuertemente de la selección de bandas de frecuencia para la extracción del ritmo sensoriomotor. Por ello, variantes como Filter Bank CSP (FBCSP) aplican el método sobre múltiples bandas de frecuencia antes de seleccionar las más discriminativas. La implementación en Python se realiza con mne.decoding.CSP de la librería MNE. [2]

**- Funcionamiento Matemático:** El método calcula matrices de covarianza para cada clase (por ejemplo MI_R y MI_L) y resuelve un problema de autovalores generalizados:

$$\boxed{C_1 ​W = \lambda C_2 W}$$

donde:
 - C1 y C2 representan las matrices de covarianza de las dos clases
 - W corresponde a los filtros espaciales
 - $\lambda$ representa la capacidad discriminativa del filtro

Los filtros obtenidos generan nuevas señales espaciales donde las diferencias entre clases son más evidentes.

**- Interpretación Fisiológica:** En tareas de imaginación motora:

- El canal C3 suele activarse durante imaginación de movimiento de la mano derecha
- El canal C4 suele activarse durante imaginación de movimiento de la mano izquierda

El CSP amplifica estas diferencias espaciales, permitiendo identificar patrones de lateralización cerebral.

**- Implementación en Python:** Se utiliza la librería MNE

    from mne.decoding import CSP

    csp = CSP(n_components=4)
    X_csp = csp.fit_transform(X, y)

**- Ventajas:**

- Alta capacidad discriminativa
- Muy eficiente en BCI motoras
- Reduce dimensionalidad
- Mejora precisión de clasificadores

#### **Parámetros de Hjorth**

Los parámetros de Hjorth son características basadas en la varianza de las derivadas de la señal EEG. Las tres primeras derivadas de la señal dan lugar a las medidas de actividad (Activity), movilidad (Mobility) y complejidad (Complexity), que son los parámetros de Hjorth frecuentemente empleados en sistemas BCI. Estos tres descriptores operan enteramente en el dominio del tiempo, lo que los hace ortogonales a la PSD y muy rápidos de calcular. [3]

Activity mide la varianza cuadrática de la amplitud de la señal EEG en una ventana considerada. Mobility se define como la potencia promedio de la derivada normalizada de la señal EEG. Complexity representa la relación entre las movilidades, es decir, la movilidad de la primera derivada dividida entre la movilidad de la señal original. En términos fisiológicos, Activity es análoga a la potencia total integrada de la PSD, Mobility refleja desplazamientos en la frecuencia media dominante del ritmo mu o beta, y Complexity captura cuánto se aleja la señal de una oscilación sinusoidal pura, lo que aumenta cuando la corteza motora se activa y la señal se vuelve más irregular. [4]

Según resultados recientes, la actividad, movilidad y complejidad de segmentos EEG captados por los canales C3, Cz y C4, con y sin descomposición en sub-bandas, proporcionaron precisiones de clasificación confiables. El enfoque propuesto permitió discriminar tareas de movimiento ejecutado e imaginado, alcanzando una precisión promedio de clasificación de 89.7 ± 0.78%. En estudios comparativos de extracción de características para BCI, se ha reportado que la combinación de potencia de bandas, parámetros de Hjorth y coeficientes autorregresivos adaptativos presenta resultados consistentes en múltiples datasets EEG de BCI. Los parámetros de Hjorth se implementan directamente con NumPy a partir de las derivadas discretas de la señal, sin necesidad de librerías especializadas. [5]

#### **Energía y Entropía Wavelet (DWT)**

La Transformada Wavelet Discreta (DWT) ofrece una representación tiempo-frecuencia de resolución múltiple que supera a la PSD convencional en señales no estacionarias como el EEG. Se propone un método de extracción de características basado en la transformada wavelet discreta (DWT), la descomposición en modos empíricos (EMD) y la entropía aproximada. La señal EEG se descompone en una serie de señales de banda estrecha con la DWT, y la entropía aproximada de la señal reconstruida se obtiene como el vector de características correspondiente. [5]

De la DWT se derivan dos índices especialmente útiles para BCI. El primero es la energía relativa por sub-banda, que representa la fracción de la energía total contenida en cada nivel wavelet, siendo directamente comparable con la PSD integrada por banda del Proyecto 1 pero con mejor localización temporal. El segundo es la entropía de Shannon wavelet, que mide qué tan distribuida está la energía entre las sub-bandas: la entropía wavelet relativa y las características topológicas de la red de función cerebral pueden extraerse de forma conjunta, logrando una precisión de clasificación promedio superior al 90% en datasets públicos de BCI, lo que demuestra que estas características retienen información de la señal EEG de forma más efectiva y reducen la complejidad computacional. [5]

En el proceso de extracción de características es posible calcular energía, varianza y entropía de la transformada wavelet basadas en cinco sub-bandas de frecuencia EEG como características en el dominio tiempo-frecuencia, las cuales, combinadas con parámetros en el dominio del tiempo y del dominio de la frecuencia, permiten obtener los mejores resultados de selección en clasificaciones binarias y multiclase de imaginería motora. La implementación en Python se realiza con la librería PyWavelets (pywt), que permite la descomposición DWT con wavelets como db4, cuyos niveles se alinean con las bandas delta, theta, mu y beta para una frecuencia de muestreo de 160 Hz.[6]

#### **Coherencia**
El análisis de coherencia en el electroencefalograma (EEG) es una métrica neurofisiológica que evalúa la conectividad funcional y la sincronización entre distintas regiones del cerebro. Actúa como un índice que mide qué tan coordinadamente se comunican y oscilan dos áreas corticales a través de bandas de frecuencia específicas.

La coherencia es una medida de conectividad funcional que cuantifica el grado de sincronización entre dos señales EEG en una banda de frecuencia específica.Esta métrica permite evaluar cómo interactúan distintas regiones cerebrales durante tareas cognitivas o motoras.[7]

La coherencia toma valores entre:

0 → sin relación
1 → sincronización perfecta

**- Funcionamiento Matemático:** La coherencia entre dos señales x(t) e y(t) se define como:

$$\boxed{C_{xy}(f) = \frac{|P_{xy}(f)|^2}{P_{xx}(f) \cdot P_{yy}(f)}}$$

donde:

- P_xy(f): densidad espectral cruzada
- P_yy(f), P_xx(f) : densidades espectrales de potencia individuales

**- Interpretación Fisiológica:** En tareas motoras:

Aumentos de coherencia pueden indicar comunicación funcional entre áreas motoras
Disminuciones pueden reflejar especialización o lateralización de la actividad

La coherencia es especialmente útil para estudiar:

conectividad cortical
coordinación interhemisférica
organización funcional cerebral

**- Implementación en Python:** Se utilizan las librerias MNE y SciPy

    from scipy.signal import coherence

    f, coh = coherence(signal1, signal2, fs=fs)

**. Ventajas**

- Evalúa interacción entre regiones cerebrales
- Complementa análisis espectral
- Permite estudiar redes neuronales funcionales

#### **Entropía Espectral**

La entropía espectral en el Electroencefalograma (EEG) es una métrica de la teoría de la información que cuantifica la irregularidad o complejidad de las señales cerebrales. Se calcula a partir de la distribución de potencia en diferentes frecuencias; valores altos indican un cerebro despierto y activo, mientras que valores bajos reflejan regularidad y menor actividad.

La entropía espectral es una medida de complejidad basada en la distribución de energía de la señal EEG en el dominio de la frecuencia.Esta métrica cuantifica el grado de desorganización o irregularidad de la actividad cerebral.

Valores bajos → actividad más regular y organizada
Valores altos → actividad más compleja y distribuida

**- Fundamento Matemático:** La entropía espectral se calcula aplicando la entropía de Shannon sobre la PSD normalizada:

$$\boxed{H = -\sum_{i=1}^{N}p_i \log(p_i)}$$

donde:

- pi: representa la distribución normalizada de potencia espectral
- H: corresponde a la entropía espectral

**- Interpretación Fisiológica:** En EEG

Estados de reposo suelen presentar patrones más organizados
Tareas cognitivas o motoras incrementan la complejidad neuronal

La entropía espectral permite detectar cambios dinámicos en la organización cortical

**- Implementación en Python:** Se utilizan las librerias AntroPy y NumPy

    import antropy as ant

    entropy = ant.spectral_entropy(signal, sf=fs)

**. Ventajas**

- Captura complejidad neuronal
- Sensible a cambios cognitivos
- Complementa PSD y conectividad

#### **Selección de Índices**

Se seleccionan los índices **Common Spatial Patterns (CSP), coherencia y entropía espectral** para implementar en el análisis de las señales EEG.

La inclusión de estos índices se justifica debido a que la densidad espectral de potencia (PSD), aunque permite identificar cambios en ritmos cerebrales asociados a tareas motoras, no captura completamente la complejidad espacial y funcional de la actividad cerebral. El método CSP aporta información espacial relevante al resaltar patrones de activación cortical que diferencian la imaginación motora de la mano derecha e izquierda, mejorando la discriminación entre condiciones. Por su parte, la coherencia permite evaluar la conectividad funcional entre regiones cerebrales motoras, proporcionando información sobre la sincronización neuronal durante las tareas de imaginación de movimiento. Finalmente, la entropía espectral complementa el análisis cuantificando la complejidad e irregularidad de la señal EEG, permitiendo detectar cambios en la organización cortical entre estados de reposo y actividad motora imaginada. En conjunto, estos índices ofrecen una representación más completa de la dinámica cerebral, mejorando la caracterización y clasificación de estados en sistemas BCI no invasivos.


</div>

### **2.PLAN DE ANÁLISIS**

Con la información recolectada, del entregable 1 y del punto anterior, proponer una metodología de análisis que permita evidenciar la diferencia en ritmos cerebrales asociados a las condiciones de: reposo, imaginación de movimiento mano derecha, imaginación de movimiento mano izquierda. 

Con base en el análisis realizado en el entregable anterior y en los índices adicionales consultados, se propone la siguiente metodología para identificar diferencias en la actividad cerebral asociada a:

reposo
imaginación de movimiento mano derecha (MI_R)
imaginación de movimiento mano izquierda (MI_L)

**1. Segmentación de señales (Epoching)**

Las señales EEG se segmentarán utilizando los marcadores de eventos del dataset:

MI_R
MI_L
Reposo

Cada época incluirá:

ventana pre-estímulo
ventana activa

Esto permitirá comparar actividad cerebral entre condiciones específicas.

**2. Extracción de características** Se calcularán múltiples índices EEG:

**Common Spatial Patterns (CSP)**

Objetivo:maximizar discriminación entre MI_R y MI_L.

Resultado esperado:patrones espaciales diferenciados entre hemisferios.

**Coherencia**

Objetivo:evaluar conectividad funcional entre regiones motoras.

Canales:

C3–C4
C3–Cz
C4–Cz

**Entropía espectral**

Objetivo: analizar complejidad cortical entre condiciones.


**3. Análisis Estadístico** Se calcularán:

media
mediana
desviación estándar
IQR

Visualizaciones:

boxplots
mapas de calor
matrices de conectividad
Esta es la etapa central de la metodología propuesta. Se calculan los tres índices sobre cada época y canal, construyendo un vector de características por época que combina información espectral (heredada del Proyecto 1), espacial, temporal y tiempo-frecuencial.

**4. Pruebas de hipótesis**

**Hipótesis principales**

H0: No existen diferencias significativas entre reposo, MI_R y MI_L en los índices EEG calculados.

H1: Existen diferencias significativas entre las condiciones, especialmente en ritmos mu y patrones espaciales motores.

**Pruebas Estadísticas** Dependiendo de la normalidad se aplicará:

ANOVA de medidas repetidas
t-test pareado
Wilcoxon
Kruskal-Wallis

**PROYECTO - MANEJO DEL DATAFRAME**

<div align =="justify">

Para el proyecto, cambiar las etiquetas en el Datframe de "Mano derecha" a valores numericos 

Se normalizan las caracteristicas, no influye la etiqueta, se normalizan todas las caracteristicas para qie qieden en un solo rango.

Se debe bajar el valor de la correlación entre las carcteristicas lo ideal es que sean casi nulas. realizar el analisis de correlación la alta correlación puede ser motiov de deficiencia del modelo.

El resultado de la normalización de los valores de caracteristicas es el array x de caracteristicas y el y es el de las etiquetas

Se deben dividir los datos en 80-20 , 80% de los datos para entrenar el modelo y el 20% pra evaluarlo o validarlo


Con train_test_split obtenemos cuatro vectores, los datos de entrenamiento, validación, etiquetas y...

La **matriz de confusión** permite identificar cuántos datos clasifica de forma correcta. Es decir, da información de los falsos negativos y positivos, aquellos valores ue pertenecía a una clase pero el modelo los calificó como si fueran de la otra clase.

Especificidad y sensibilidad, para decidir ante los resultados que de el modelo. 

Es importane que el modelo reciba datos proporcionales, igual cantidad de datos de cada clase.

**Clasification_report** Compara las etiquetas predichas con las reales. Se hace para entrenamiento y evaluación 

</div>

### **REFERENCIAS**


[1] Blankertz, B., Tomioka, R., Lemm, S., Kawanabe, M., & Müller, K.-R. (2008). Optimizing spatial filters for robust EEG single-trial analysis. IEEE Signal Processing Magazine, 25(1), 41–56. https://doi.org/10.1109/MSP.2008.4408441

[2] Degirmenci, M., Yuce, Y. K., Perc, M., & Isler, Y. (2023). Statistically significant features improve binary and multiple Motor Imagery task predictions from EEGs. Frontiers in Human Neuroscience, 17, 1223307. https://doi.org/10.3389/fnhum.2023.1223307

[3] Elahi, M. H. et al. (2023). EEG-BCI Features Discrimination between Executed and Imagined Movements Based on FastICA, Hjorth Parameters, and SVM. Mathematics, 11(21), 4409. https://doi.org/10.3390/math11214409

[4] Garcia-Laencina, P. J. et al. (2014). Exploring dimensionality reduction of EEG features in motor imagery task classification. Expert Systems with Applications, 41(10), 4761–4771. https://doi.org/10.1016/j.eswa.2014.02.021

[5] Ji, N., Ma, L., Dong, H., & Zhang, X. (2019). EEG Signals Feature Extraction Based on DWT and EMD Combined with Approximate Entropy. Brain Sciences, 9(8), 201. https://doi.org/10.3390/brainsci9080201

[6] Wang, M. et al. (2023). Motor imagery classification method based on relative wavelet packet entropy brain network and improved lasso. Frontiers in Neuroscience, 17, 1113593. https://doi.org/10.3389/fnins.2023.1113593

[7] M. Atienza, J.L. Cantero-Lorente, R.M. Salas. Valor clínico de la coherencia EEG como índice electrofisiológico de conectividad córticocortical durante el sueño. Rev. Neurol. 2000, 31(5), 442-454. https://doi.org/10.33588/rn.3105.99472
 
[8] MNE Documentation: https://mne.tools
